
# Multivariate Forecasting

This section defines the key variables required for building a multivariate forecasting model for power generation.

***1. Fixed Variables***

These variables remain constant over time and describe the static characteristics of each plant.

- **`plant_id`**: Unique identifier for the power plant.
    
- **`plant_type`**: Type of plant (e.g., solar, wind, hydro, thermal).
    
- **`capacity_mw`**: Installed capacity of the plant in megawatts (MW).
    

***2. Weather Information***

These variables capture environmental conditions that directly influence power generation.

- **`temperature`**: Ambient temperature in degrees Celsius (°C).
    
- **`wind_speed`**: Wind speed in meters per second (m/s).
    
- **`wind_direction`**: Direction of wind in degrees (0–360°) or cardinal directions (e.g., N, NE, SW).
    
- **`irradiance`**: Solar irradiance in watts per square meter (W/m²).
    
- **`cloud_cover`**: Fraction or percentage of sky covered by clouds (0–100%).
    

***3. Plant-Based Forecast Inputs***

These variables represent operational and forecast-related parameters specific to each plant.

- **`forecast_irradiance`**: Predicted solar irradiance used for generation forecasting (W/m²).
    
- **`health_factor`**: Efficiency indicator of the plant (range: 0–1), accounting for degradation, maintenance, and system losses.
    
- **`availability`**: Fraction of time the plant is operational and capable of generating power (0–1).
    
- **`curtailment`**: Reduction in generation due to grid constraints or external instructions (in MW or percentage).
    


***4. Target Variable***

To complete the forecasting setup, include a target variable:

- **`actual_generation_mw`**: Actual generated power output in megawatts (MW).
    



----

# Univariate Generation Forecast

This section defines the minimal set of variables required for univariate power generation forecasting, along with its purpose and limitations.

 ***1. Required Variables***

- **`timestamp`**: Date-time of the observation (critical for time-series forecasting).
    
- **`plant_id`**: Unique identifier for the power plant.
    
- **`plant_type`**: Type of plant (e.g., solar, wind, hybrid).
    
- **`actual_generation_mw`**: Actual power generated by the plant in megawatts (MW).
    

***2. Overview***

Univariate forecasting relies solely on historical generation data to predict future output. It does not explicitly incorporate external influencing factors such as weather or operational conditions.

While simpler to implement and maintain, this approach may not capture real-world variability driven by environmental and plant-specific parameters.

***3. Use Cases***

Univariate forecasting is primarily used for the following purposes:

- **Curtailment Estimation (Baseline Reference)**  
    Provides a baseline generation estimate against which curtailment can be approximated when actual output deviates.
    
- **System Backup / Fallback Model**  
    Acts as a lightweight and reliable backup for production systems (e.g., website or APIs) in case multivariate models fail or data inputs are unavailable.
    

 ***4. Limitations***

- Does not account for weather dependencies (e.g., irradiance, wind speed).
    
- Cannot capture plant health, availability, or operational constraints.
    
- Lower accuracy compared to multivariate forecasting in most real-world scenarios.

***5. Model Candidates*** 

The forecasting pipeline evaluates multiple time-series and machine learning models. The best-performing model is selected and refit on the full historical series before generating forecasts.

*Statistical Models*

- **ETS (Error-Trend-Seasonality)**: Captures level, trend, and seasonality components.
- **Theta Model**: Effective for strong seasonality and trend decomposition.
- **SARIMA**: Handles autoregression, differencing, and seasonal patterns.
- **Prophet**: Decomposable model with trend, seasonality, and holiday effects.

*Machine Learning Models*

- **LightGBM**: Gradient boosting model optimized for speed and performance on tabular time-series features.
- **XGBoost**: Robust boosting algorithm with strong performance on structured data.

*Deep Learning Model (Conditional Use)*

- **LSTM (Long Short-Term Memory)**:  
    Used **only for hybrid plant types if required**, where generation patterns are more complex due to multiple energy sources (e.g., solar + wind). Helps capture long-term temporal dependencies that simpler models may miss.

***6. Forecast Output***

For the selected best model:

- The model is **refit on the full time series**.
- Forecasts are generated for the defined horizon (e.g., next _N_ hours).
- Prediction intervals are also computed:
    - Statistical models: derived from model confidence intervals (if available).
    - ML/DL models: approximated using RMSE-based bounds.
- **Outputs include**:
    - `forecast_timestamps`
    - `forecast_mw`
    - `forecast_lower_bound`
    - `forecast_upper_bound`

In [1]:
import pandas as pd

plant = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\plant_master.csv")
generation_historical = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\generation.csv", parse_dates=["timestamp"])


# Merge plant with generation on 'plant_id'
merged_df = plant.merge(generation_historical, on=['plant_id', 'plant_type' ,'region'], how='left')

df = merged_df[['timestamp', 'plant_id', 'plant_type', 'actual_generation_mw']]

df.head()

,timestamp,plant_id,plant_type,actual_generation_mw
0,2024-01-01 00:00:00,PLANT_001,Solar,0.0
1,2024-01-01 01:00:00,PLANT_001,Solar,0.0
2,2024-01-01 02:00:00,PLANT_001,Solar,0.0
3,2024-01-01 03:00:00,PLANT_001,Solar,0.0
4,2024-01-01 04:00:00,PLANT_001,Solar,0.0


In [2]:
from src.generation_univariate import prepare_univariate_input, run_univariate_fleet
univ_input = prepare_univariate_input(df)


[Prep] Ready for univariate forecasting:
  Plants included : 50
  Plants skipped  : 0
  Total rows      : 574,850
  Columns         : ['timestamp', 'plant_id', 'plant_type', 'generation']
  Date range      : 2024-01-01 00:00:00 → 2025-04-24 00:00:00

  Rows per plant type:
plant_type
Hybrid     4
Solar     25
Wind      21


In [3]:
# Run — no other changes needed in run_univariate_fleet
forecast_df = run_univariate_fleet(
    fleet_df   = univ_input,
    horizon    = 72,
    n_jobs     = -1,
)


════════════════════════════════════════════════════════════
  FLEET UNIVARIATE FORECAST
  Plants   : 50  (solar=0  wind=0  hybrid=0)
  Horizon  : 72h
  Workers  : 8 / 8 cores  (n_jobs=-1)
  Backend  : loky
  LSTM threshold : 15.0 MAE (hybrid only)
════════════════════════════════════════════════════════════



[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done   2 tasks      | elapsed:  6.8min
[Parallel(n_jobs=8)]: Done   9 tasks      | elapsed: 12.0min
[Parallel(n_jobs=8)]: Done  16 tasks      | elapsed: 22.1min
[Parallel(n_jobs=8)]: Done  25 tasks      | elapsed: 63.1min
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed: 68.1min
[Parallel(n_jobs=8)]: Done  41 out of  50 | elapsed: 74.3min remaining: 16.3min
[Parallel(n_jobs=8)]: Done  47 out of  50 | elapsed: 81.4min remaining:  5.2min
[Parallel(n_jobs=8)]: Done  50 out of  50 | elapsed: 106.5min finished


  ✓ PLANT_001 (solar) — ets  MAE=1.889
  ✓ PLANT_002 (wind) — theta  MAE=7.616
  ✗ PLANT_003 (hybrid) — ERROR: UnboundLocalError: cannot access local variable 'fit_obj' where it is not associated with a value
  ✓ PLANT_004 (solar) — ets  MAE=2.971
  ✓ PLANT_005 (wind) — ets  MAE=6.252
  ✓ PLANT_006 (solar) — ets  MAE=0.356
  ✗ PLANT_007 (wind) — ERROR: UnboundLocalError: cannot access local variable 'fit_obj' where it is not associated with a value
  ✓ PLANT_008 (solar) — ets  MAE=0.942
  ✓ PLANT_009 (hybrid) — sarima  MAE=3.244
  ✓ PLANT_010 (solar) — theta  MAE=2.711
  ✗ PLANT_011 (wind) — ERROR: MemoryError: Unable to allocate 87.7 MiB for an array with shape (1000, 11497) and data type float64
  ✓ PLANT_012 (solar) — theta  MAE=2.934
  ✗ PLANT_013 (hybrid) — ERROR: MemoryError: Unable to allocate 219. MiB for an array with shape (50, 50, 11498) and data type float64
  ✓ PLANT_014 (solar) — ets  MAE=1.382
  ✓ PLANT_015 (wind) — ets  MAE=5.593
  ✓ PLANT_016 (solar) — ets  MAE=0.377
 

---


# Univariate Irradiance Forecasting

 ***1. Required Variables***

- **`timestamp`**: Date-time of the observation (critical for time-series forecasting).
    
- **`plant_id`**: Unique identifier for the power plant.
    
- **`plant_type`**: Type of plant (e.g., solar, wind, hybrid).
    
- **`irradiance`**: Represents **solar radiation intensity** (typically W/m²).
    

***2. Overview***

Irradiance forecasting focuses on predicting the amount of solar radiation reaching the Earth's surface. It is a key component in solar power generation forecasting.

At its core:

> **Irradiance = Solar Geometry + Cloud Patterns**

- **Solar Geometry** determines the position of the sun based on time and location.
- **Cloud Patterns** influence how much sunlight actually reaches the surface.


***3. Model Candidates*** 
Why Prophet for Irradiance?

- Captures **strong daily and seasonal patterns** of sunlight.
- Handles **non-linear trends** caused by weather variability.
- Easy to incorporate **additional regressors** (e.g., cloud cover, temperature).

In [3]:
import pandas as pd

plant = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\plant_master.csv")
generation_historical = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\generation.csv", parse_dates=["timestamp"])
generation_historical = generation_historical[['plant_id', 'timestamp', 'irradiance_wm2']]

In [4]:
from src.irradiance import run_irradiance_fleet

# Run both auxiliary forecasts
irradiance_fc = run_irradiance_fleet(generation_historical, plant, n_jobs=-1)


[Irradiance prep] 50 plants | 574,850 rows | daytime mean: 457.5 W/m²

═══════════════════════════════════════════════════════
  IRRADIANCE FLEET FORECAST
  Plants: 50 | Workers: 8 | Horizon: 72h
═══════════════════════════════════════════════════════


[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done   2 tasks      | elapsed: 14.3min
[Parallel(n_jobs=8)]: Done  46 out of  50 | elapsed: 88.0min remaining:  7.7min
[Parallel(n_jobs=8)]: Done  50 out of  50 | elapsed: 92.0min finished


  ✓ PLANT_001 → prophet
  ✓ PLANT_002 → prophet
  ✓ PLANT_003 → prophet
  ✓ PLANT_004 → prophet
  ✓ PLANT_005 → prophet
  ✓ PLANT_006 → prophet
  ✓ PLANT_007 → prophet
  ✓ PLANT_008 → prophet
  ✓ PLANT_009 → prophet
  ✓ PLANT_010 → prophet
  ✓ PLANT_011 → prophet
  ✓ PLANT_012 → prophet
  ✓ PLANT_013 → prophet
  ✓ PLANT_014 → prophet
  ✓ PLANT_015 → prophet
  ✓ PLANT_016 → prophet
  ✓ PLANT_017 → prophet
  ✓ PLANT_018 → prophet
  ✓ PLANT_019 → prophet
  ✓ PLANT_020 → prophet
  ✓ PLANT_021 → prophet
  ✓ PLANT_022 → prophet
  ✓ PLANT_023 → prophet
  ✓ PLANT_024 → prophet
  ✓ PLANT_025 → prophet
  ✓ PLANT_026 → prophet
  ✓ PLANT_027 → prophet
  ✓ PLANT_028 → prophet
  ✓ PLANT_029 → prophet
  ✓ PLANT_030 → prophet
  ✓ PLANT_031 → prophet
  ✓ PLANT_032 → prophet
  ✓ PLANT_033 → prophet
  ✓ PLANT_034 → prophet
  ✓ PLANT_035 → prophet
  ✓ PLANT_036 → prophet
  ✓ PLANT_037 → prophet
  ✓ PLANT_038 → prophet
  ✓ PLANT_039 → prophet
  ✓ PLANT_040 → prophet
  ✓ PLANT_041 → prophet
  ✓ PLANT_042 → 

---

#  Health Factor Forecasting

 _**1. Required Variables**_

- **`timestamp`**: Date-time of the observation (critical for time-series forecasting)
- **`plant_id`**: Unique identifier for the power plant
- **`plant_type`**: Type of plant (e.g., solar, wind, hybrid)
- **`health_factor`**: Represents the **operational health of the plant** (typically between **0.35 – 0.99**)


_**2. Overview**_

Health factor forecasting focuses on predicting the **degradation and recovery behavior** of a power plant over time.

At its core:

> **Health Factor = Degradation Trend + Repair Events**

- **Degradation Trend**
    - Slow, continuous decline in performance over time
    - Typically **−0.3% to −0.5% per month**
- **Repair Events**
    - Sudden improvements due to maintenance or component replacement
    - Causes **step increases** in health


 _**3. Model Candidates**_

Health factor is **not a typical time-series problem**. It has:

- No seasonality
- Near-linear behavior over short horizons
- Rare but impactful step changes

**Linear Regression (Primary Model)**

- Fits degradation trend over recent history (e.g., last 30 days)
- Simple, interpretable, and highly effective
- Works well because:
    - Changes over 72 hours are minimal
    - Noise is very low

In [1]:
import pandas as pd

lifecycle = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\lifecycle_events.csv")
generation_historical = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\generation.csv", parse_dates=["timestamp"])
generation_historical = generation_historical[['plant_id', 'timestamp', 'health_factor']]

In [2]:
from src.health_factor import run_health_fleet

health_fc  = run_health_fleet(generation_historical, lifecycle, n_jobs=-1)



HEALTH FORECAST | Plants: 50

✅ Saved forecasts → data\forecasts/health_forecasts.csv


---

# Availability Forecasting

***1. Required Variables***

- **`timestamp`**: Date-time of the observation (critical for time-series forecasting).

- **`plant_id`**: Unique identifier for the power plant.

- **`plant_type`**: Type of plant (e.g., solar, wind, hybrid).

- **`availability_mw`**: Represents **available generation capacity** (in MW).

- **`capacity_mw`**: Installed capacity of the plant (upper bound).


***2. Overview***

Availability forecasting focuses on predicting the **operational readiness of a power plant**, i.e., how much of its installed capacity is available for generation.

At its core:

> **Availability = Capacity − Outages (Planned + Unplanned)**

- **Planned Maintenance**
  - Scheduled downtime for inspection, servicing, or upgrades
  - Often follows **weekly or seasonal patterns**

- **Forced Outages**
  - Unexpected failures (equipment faults, grid issues)
  - Cause **sudden drops to partial or zero availability**


***3. Model Candidates***

**Why Prophet for Availability?**

- Captures **weekly patterns** (e.g., weekend maintenance schedules)
- Models **seasonality** (e.g., higher maintenance during low-demand periods like monsoon)
- Handles **trend + changepoints** for outage behavior
- Supports **bounded forecasts** using logistic growth (≤ capacity)


**Capacity Baseline (Critical Component)**

- Assume **full availability = capacity_mw**
- Acts as a **physical upper bound**
- Very strong baseline since plants operate near full capacity most of the time


**Blended Approach (Recommended)**

> **Final Forecast = 0.7 × Prophet + 0.3 × Capacity Baseline**

- Prophet captures outage patterns  
- Baseline ensures **stability and realism**  
- Prevents overestimation of outages  


In [2]:
from src.availability import run_availability_fleet
import pandas as pd

plant = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\plant_master.csv")
generation_historical = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\generation.csv", parse_dates=["timestamp"])

merged_df = plant.merge(generation_historical, on=['plant_id', 'plant_type' ,'region'], how='left')
merged_df = merged_df[["timestamp", "plant_id", "availability_mw", "capacity_mw"]]

availability_fc  = run_availability_fleet(
    merged_df = merged_df,
    horizon   = 72,
    n_jobs    = -1,
)

[Availability prep] 50 plants | 574,850 rows | avg utilisation: 78.7%

═══════════════════════════════════════════════════════
  AVAILABILITY FLEET FORECAST
  Plants: 50 | Workers: 8 | Horizon: 72h
═══════════════════════════════════════════════════════


[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done   2 tasks      | elapsed: 12.6min
[Parallel(n_jobs=8)]: Done  46 out of  50 | elapsed: 85.9min remaining:  7.5min
[Parallel(n_jobs=8)]: Done  50 out of  50 | elapsed: 89.1min finished


  ✓ PLANT_001 → prophet
  ✓ PLANT_002 → prophet
  ✓ PLANT_003 → prophet
  ✓ PLANT_004 → prophet
  ✓ PLANT_005 → prophet
  ✓ PLANT_006 → prophet
  ✓ PLANT_007 → prophet
  ✓ PLANT_008 → prophet
  ✓ PLANT_009 → prophet
  ✓ PLANT_010 → prophet
  ✓ PLANT_011 → prophet
  ✓ PLANT_012 → prophet
  ✓ PLANT_013 → prophet
  ✓ PLANT_014 → prophet
  ✓ PLANT_015 → prophet
  ✓ PLANT_016 → prophet
  ✓ PLANT_017 → prophet
  ✓ PLANT_018 → prophet
  ✓ PLANT_019 → prophet
  ✓ PLANT_020 → prophet
  ✓ PLANT_021 → prophet
  ✓ PLANT_022 → prophet
  ✓ PLANT_023 → prophet
  ✓ PLANT_024 → prophet
  ✓ PLANT_025 → prophet
  ✓ PLANT_026 → prophet
  ✓ PLANT_027 → prophet
  ✓ PLANT_028 → prophet
  ✓ PLANT_029 → prophet
  ✓ PLANT_030 → prophet
  ✓ PLANT_031 → prophet
  ✓ PLANT_032 → prophet
  ✓ PLANT_033 → prophet
  ✓ PLANT_034 → prophet
  ✓ PLANT_035 → prophet
  ✓ PLANT_036 → prophet
  ✓ PLANT_037 → prophet
  ✓ PLANT_038 → prophet
  ✓ PLANT_039 → prophet
  ✓ PLANT_040 → prophet
  ✓ PLANT_041 → prophet
  ✓ PLANT_042 → 


# Curtailment Forecasting

***1. Required Variables***

- **`timestamp`**: Date-time of the observation (critical for time-series forecasting).

- **`plant_id`**: Unique identifier for the power plant.

- **`plant_type`**: Type of plant (e.g., solar, wind, hybrid).

- **`curtailment_mw`**: Represents **reduced generation due to external constraints** (in MW).

- **`generation_mw`**: Actual generated power.

- **`irradiance`**: Solar radiation intensity (proxy for potential generation).

- **`capacity_mw`**: Installed capacity of the plant.


***2. Overview***

Curtailment forecasting focuses on predicting **how much potential generation is intentionally reduced** due to grid or market constraints.

At its core:

> **Curtailment = Potential Generation − Actual Generation**

- **Grid Constraints**
  - Transmission limits, congestion, stability requirements
  - Causes enforced reduction in output

- **Demand-Supply Imbalance**
  - Low demand periods (e.g., mid-day solar surplus)
  - Grid operators curtail excess generation

- **Policy / Dispatch Decisions**
  - Priority dispatch rules
  - Renewable balancing mechanisms

***3. Model Candidates***  
  
Curtailment is a **zero-inflated, event-driven problem**:  
  
- Most hours → **curtailment = 0**  
- Few hours → **sharp spikes (MW loss)**  
  
A single regression model fails because:  
- It learns to predict near-zero everywhere  
- Misses the spike structure entirely  
  
**Recommended Approach: Two-Stage Modeling**  
  
> **Stage 1 — Classification:**  
> Will curtailment occur? (Yes / No)  
  
> **Stage 2 — Regression:**  
> If yes → how much curtailment (MW)?  
  
 **Stage 1: Classification Model**  
  
- Target: `curtailed` (0/1)  
- Models:  
- **LightGBM Classifier (preferred)**  
- Gradient Boosting (fallback)  
  
**Why this works:**  
- Handles **class imbalance** (few curtailment events)  
- Learns **grid decision patterns**, not physics  
  
**Key features used:**  
- Time: hour, day-of-week, month  
- Grid signals:  
- `gen_pressure` (generation / capacity)  
- `irradiance_norm`  
- Seasonality:  
- monsoon indicator  
- peak solar hours  
- Lag features:  
- previous curtailment events  
- rolling curtailment rates  
  
 **Stage 2: Regression Model**  
  
- Target: `curtailment_mw` (only where curtailed = 1)  
- Models:  
- **LightGBM Regressor (preferred)**  
- Gradient Boosting (fallback)  
- Mean fallback if very few events  
  
**Why separate regression:**  
- Focuses only on **curtailment magnitude**  
- Avoids dilution from zero-heavy data  
  
**Final Forecast**  
  
> **Expected Curtailment = Probability × Amount**  
  
- From classifier → probability of curtailment  
- From regressor → MW given curtailment  
  
This gives:  
- Smooth expected value  
- Captures both **event likelihood + severity**  
  
**Feature Design (Critical for Performance)**  
  
Curtailment is driven by **grid behavior**, not just weather.  
  
Most important signals:  
  
- **Generation pressure**  
- High generation → higher curtailment risk  
  
- **Time patterns**  
- Midday (solar peak)  
- Weekends (lower demand)  
  
- **Seasonality**  
- Monsoon → higher curtailment (hydro competition)  
  
- **Lag features**  
- Curtailment tends to persist across hours

In [1]:
import pandas as pd

plant = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\plant_master.csv")
generation_historical = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\generation.csv", parse_dates=["timestamp"])
generation_forecast = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\forecasts\univariate_forecasts.csv", parse_dates=["timestamp"])
irradiance_forecast = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\forecasts\irradiance_forecasts.csv", parse_dates=["timestamp"])

# ── Prepare the single input dataframe ───────────────────────────────

merged_df = plant.merge(generation_historical, on=['plant_id', 'plant_type' ,'region'], how='left')
hist = merged_df[['timestamp', 'plant_id', 'plant_type', "capacity_mw",'actual_generation_mw','curtailment_mw', 'irradiance_wm2']].copy()
hist["is_forecast_row"] = 0
hist["forecast_mw"]     = hist["actual_generation_mw"]

generation_forecast.drop(columns=['lower_90', 'upper_90', 'model'], inplace=True)  
generation_forecast= generation_forecast.merge(
    plant[["plant_id", "plant_type", "capacity_mw"]],
    on="plant_id", how="left",
)
generation_forecast["is_forecast_row"]      = 1
generation_forecast["curtailment_mw"]       = 0.0
generation_forecast["actual_generation_mw"] = generation_forecast["forecast_mw"]
generation_forecast = generation_forecast.merge(
    irradiance_forecast[["timestamp", "plant_id", "irradiance_forecast"]],
    on=["timestamp", "plant_id"], how="left"
)

curtailment_input_df = pd.concat([hist, generation_forecast], ignore_index=True)
curtailment_input_df.fillna(0, inplace=True)
curtailment_input_df.head()

,timestamp,plant_id,plant_type,capacity_mw,actual_generation_mw,curtailment_mw,irradiance_wm2,is_forecast_row,forecast_mw,irradiance_forecast
0,2024-01-01 00:00:00,PLANT_001,Solar,102,0.0,0.00,0.0,0,0.0,0.0
1,2024-01-01 01:00:00,PLANT_001,Solar,102,0.0,0.00,0.0,0,0.0,0.0
2,2024-01-01 02:00:00,PLANT_001,Solar,102,0.0,2.45,0.0,0,0.0,0.0
3,2024-01-01 03:00:00,PLANT_001,Solar,102,0.0,0.00,0.0,0,0.0,0.0
4,2024-01-01 04:00:00,PLANT_001,Solar,102,0.0,0.00,0.0,0,0.0,0.0


In [2]:
from src.curtailment import run_curtailment_fleet

curtailment_forecast_df = run_curtailment_fleet(
    curtailment_input_df = curtailment_input_df,
    val_days             = 14,
    n_jobs               = -1,
)

Building curtailment features...
Building future curtailment features...
  [WARN] PLANT_003 — no forecast rows found, skipping
  [WARN] PLANT_007 — no forecast rows found, skipping
  [WARN] PLANT_011 — no forecast rows found, skipping
  [WARN] PLANT_013 — no forecast rows found, skipping
  [WARN] PLANT_017 — no forecast rows found, skipping
  [WARN] PLANT_025 — no forecast rows found, skipping
  [WARN] PLANT_027 — no forecast rows found, skipping
  [WARN] PLANT_028 — no forecast rows found, skipping
  [WARN] PLANT_031 — no forecast rows found, skipping
  [WARN] PLANT_032 — no forecast rows found, skipping
  [WARN] PLANT_033 — no forecast rows found, skipping

════════════════════════════════════════════════════════════
  FLEET CURTAILMENT FORECAST
  Plants   : 39
  Workers  : 8 / 8 cores
  Val days : 14
════════════════════════════════════════════════════════════



[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    6.4s
[Parallel(n_jobs=-1)]: Done  32 out of  39 | elapsed:    8.4s remaining:    1.8s


  ✓ PLANT_001
  ✓ PLANT_002
  ✓ PLANT_004
  ✓ PLANT_005
  ✓ PLANT_006
  ✓ PLANT_008
  ✓ PLANT_009
  ✓ PLANT_010
  ✓ PLANT_012
  ✓ PLANT_014
  ✓ PLANT_015
  ✓ PLANT_016
  ✓ PLANT_018
  ✓ PLANT_019
  ✓ PLANT_020
  ✓ PLANT_021
  ✓ PLANT_022
  ✓ PLANT_023
  ✓ PLANT_024
  ✓ PLANT_026
  ✓ PLANT_029
  ✓ PLANT_030
  ✓ PLANT_034
  ✓ PLANT_035
  ✓ PLANT_036
  ✓ PLANT_037
  ✓ PLANT_038
  ✓ PLANT_039
  ✓ PLANT_040
  ✓ PLANT_041
  ✓ PLANT_042
  ✓ PLANT_043
  ✓ PLANT_044
  ✓ PLANT_045
  ✓ PLANT_046
  ✓ PLANT_047
  ✓ PLANT_048
  ✓ PLANT_049
  ✓ PLANT_050

Curtailment forecasts saved → data\curtailment/curtailment_forecasts.csv


[Parallel(n_jobs=-1)]: Done  39 out of  39 | elapsed:    8.8s finished


Building curtailment features...
Building future curtailment features...
  [WARN] PLANT_013 — no forecast rows found, skipping
  [WARN] PLANT_015 — no forecast rows found, skipping
  [WARN] PLANT_024 — no forecast rows found, skipping

════════════════════════════════════════════════════════════
  FLEET CURTAILMENT FORECAST
  Plants   : 47
  Workers  : 8 / 8 cores
  Val days : 14
════════════════════════════════════════════════════════════



[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    8.4s
[Parallel(n_jobs=-1)]: Done  42 out of  47 | elapsed:   11.1s remaining:    1.2s
[Parallel(n_jobs=-1)]: Done  47 out of  47 | elapsed:   11.4s finished


  ✓ PLANT_001
  ✓ PLANT_002
  ✓ PLANT_003
  ✓ PLANT_004
  ✓ PLANT_005
  ✓ PLANT_006
  ✓ PLANT_007
  ✓ PLANT_008
  ✓ PLANT_009
  ✓ PLANT_010
  ✓ PLANT_011
  ✓ PLANT_012
  ✓ PLANT_014
  ✓ PLANT_016
  ✓ PLANT_017
  ✓ PLANT_018
  ✓ PLANT_019
  ✓ PLANT_020
  ✓ PLANT_021
  ✓ PLANT_022
  ✓ PLANT_023
  ✓ PLANT_025
  ✓ PLANT_026
  ✓ PLANT_027
  ✓ PLANT_028
  ✓ PLANT_029
  ✓ PLANT_030
  ✓ PLANT_031
  ✓ PLANT_032
  ✓ PLANT_033
  ✓ PLANT_034
  ✓ PLANT_035
  ✓ PLANT_036
  ✓ PLANT_037
  ✓ PLANT_038
  ✓ PLANT_039
  ✓ PLANT_040
  ✓ PLANT_041
  ✓ PLANT_042
  ✓ PLANT_043
  ✓ PLANT_044
  ✓ PLANT_045
  ✓ PLANT_046
  ✓ PLANT_047
  ✓ PLANT_048
  ✓ PLANT_049
  ✓ PLANT_050

Curtailment forecasts saved → data\curtailment/curtailment_forecasts.csv


In [16]:
curtailment_forecast_df.head()

,plant_id,timestamp,curtailment_mw_forecast,curtail_probability,curtail_amount_given_event
0,PLANT_001,2025-04-24 01:00:00,0.121,0.078,1.553
1,PLANT_001,2025-04-24 02:00:00,0.121,0.078,1.553
2,PLANT_001,2025-04-24 03:00:00,0.140,0.090,1.553
3,PLANT_001,2025-04-24 04:00:00,0.140,0.090,1.553
4,PLANT_001,2025-04-24 05:00:00,0.140,0.090,1.553


In [17]:
final_forecast_df.head()

,plant_id,timestamp,gross_forecast_mw,lower_90,upper_90,model,curtailment_mw_forecast,curtail_probability,net_forecast_mw
0,PLANT_001,2025-04-24 01:00:00,0.0,0.0,0.0,ets,0.121,0.078,0.0
1,PLANT_001,2025-04-24 02:00:00,0.0,0.0,0.0,ets,0.121,0.078,0.0
2,PLANT_001,2025-04-24 03:00:00,0.0,0.0,0.0,ets,0.140,0.090,0.0
3,PLANT_001,2025-04-24 04:00:00,0.0,0.0,0.0,ets,0.140,0.090,0.0
4,PLANT_001,2025-04-24 05:00:00,0.0,0.0,0.0,ets,0.140,0.090,0.0


In [18]:
curtailment_forecast_df.to_csv(r"D:\Portfolio Github\Pravaah\data\forecasts\curtailment_forecasts.csv", index=False)
final_forecast_df.to_csv(r"D:\Portfolio Github\Pravaah\data\forecasts\final_univariate_forecasts.csv", index=False)

# Multivariate Generation Forecasting

## Importing Forecast Data 

- Plant
- Weather 
- availability
- curtailment 
- irradiance
- health factor

In [1]:
import pandas as pd

plant = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\plant_master.csv")
weather_forecast = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\weather_forecast.csv", parse_dates=["timestamp"])
curtailment_forecast = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\forecasts\curtailment_forecasts.csv", parse_dates=["timestamp"])
irradiance_forecast = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\forecasts\irradiance_forecasts.csv", parse_dates=["timestamp"])
health_forecast = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\forecasts\health_forecasts.csv", parse_dates=["timestamp"])
availability_forecast = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\forecasts\availability_forecasts.csv", parse_dates=["timestamp"])

# Merge plant with generation on 'plant_id'
merged_df = plant.merge(curtailment_forecast, on=['plant_id'], how='left')

# Merge with weather on 'plant_id' and 'timestamp'
merged_df = merged_df.merge(weather_forecast, on=['plant_name', 'timestamp'], how='left')

# Merge with irradiance on 'plant_id' and 'timestamp'
merged_df = merged_df.merge(irradiance_forecast, on=['plant_id', 'timestamp'], how='left')

# Merge with health on 'plant_id' and 'timestamp'
merged_df = merged_df.merge(health_forecast, on=['plant_id', 'timestamp'], how='left')

# Merge with availability on 'plant_id' and 'timestamp'
merged_df = merged_df.merge(availability_forecast, on=['plant_id', 'timestamp'], how='left')

forecast_df = merged_df.drop(columns=[ 'plant_name', 'curtail_probability','lower_90',
       'upper_90','lower_90_x', 'upper_90_x','lower_90_y', 'upper_90_y', 'method_x','repair_probability',
    'latitude', 'longitude', 'commission_date', 'region', 'state',
    'developer', 'offtaker','method_y'
])

forecast_df.rename(columns={
    "curtailment_mw_forecast": "curtailment_mw",
    'availability_forecast': "availability_mw",
    "irradiance_forecast": "irradiance_wm2",
    "health_forecast": "health_factor"
}, inplace=True)

forecast_df.columns


Index(['plant_id', 'plant_type', 'capacity_mw', 'timestamp', 'curtailment_mw',
       'curtail_amount_given_event', 'temperature', 'cloud_cover',
       'wind_speed', 'wind_direction', 'irradiance', 'irradiance_wm2',
       'health_factor', 'availability_mw'],
      dtype='str')

## Importing Historical Data 

- Plant
- Generation
- Weather

In [2]:
import pandas as pd

plant = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\plant_master.csv")
weather_historical = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\weather_historical.csv", parse_dates=["timestamp"])
generation_historical = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\generation.csv", parse_dates=["timestamp"])


# Merge plant with generation on 'plant_id'
merged_df = plant.merge(generation_historical, on=['plant_id', 'plant_type' ,'region'], how='left')

# Merge with weather on 'plant_id' and 'timestamp'
merged_df = merged_df.merge(weather_historical, on=['plant_name', 'timestamp'], how='left')

historical_df = merged_df.drop(columns=[ 'plant_name', 
    'latitude', 'longitude', 'commission_date', 'region', 'state',
    'developer', 'offtaker','status',
])

historical_df.columns

Index(['plant_id', 'plant_type', 'capacity_mw', 'timestamp',
       'actual_generation_mw', 'availability_mw', 'curtailment_mw',
       'health_factor', 'irradiance_wm2', 'temperature', 'cloud_cover',
       'wind_speed', 'wind_direction', 'irradiance'],
      dtype='str')

## Pre-processing

In [3]:
from src.preprocessing import preprocess

historical_df = preprocess(historical_df)
forecast_df = preprocess(forecast_df,is_forecast=True)


────────────────────────────────────────────────────────────
  STARTING PRE-PROCESSING PIPELINE
────────────────────────────────────────────────────────────
[INFO] irradiance_wm2 vs irradiance correlation: 0.6111
[OK] Merged irradiance_wm2 + irradiance → single 'irradiance' column
[OK] Numeric dtypes enforced
[OK] Imputation complete — nulls reduced: 919,880 → 229,940
[WARN] Remaining nulls:
temperature    229940
dtype: int64
[OK] Outlier treatment done (iqr) on: ['actual_generation_mw', 'availability_mw', 'temperature', 'cloud_cover', 'wind_speed', 'irradiance']
[OK] Dtypes optimised — memory usage: 47.1 MB

════════════════════════════════════════════════════════════
  PRE-PROCESSING QA REPORT
════════════════════════════════════════════════════════════
  Shape          : 574,850 rows × 14 columns
  Plants         : 50
  Plant types    : {'Solar': 287425, 'Wind': 241437, 'Hybrid': 45988}
  Date range     : 2024-01-01 00:00:00 → 2025-04-24 00:00:00
  Null values    : 229940
  Duplica

In [6]:
historical_df.columns

Index(['plant_id', 'plant_type', 'capacity_mw', 'timestamp',
       'actual_generation_mw', 'availability_mw', 'curtailment_mw',
       'health_factor', 'temperature', 'cloud_cover', 'wind_speed',
       'wind_direction', 'irradiance', 'generation'],
      dtype='str')

In [7]:
forecast_df.columns

Index(['plant_id', 'plant_type', 'capacity_mw', 'timestamp', 'curtailment_mw',
       'curtail_amount_given_event', 'temperature', 'cloud_cover',
       'wind_speed', 'wind_direction', 'irradiance', 'health_factor',
       'availability_mw'],
      dtype='str')

## Categorize the data

In [4]:
historical_df_solar = historical_df[historical_df['plant_type'] == 'Solar'].copy()
historical_df_wind = historical_df[historical_df['plant_type'] == 'Wind'].copy()
historical_df_hybrid = historical_df[historical_df['plant_type'] == 'Hybrid'].copy()    

forecast_df_solar = forecast_df[forecast_df['plant_type'] == 'Solar'].copy()
forecast_df_wind = forecast_df[forecast_df['plant_type'] == 'Wind'].copy()
forecast_df_hybrid = forecast_df[forecast_df['plant_type'] == 'Hybrid'].copy() 

In [7]:
historical_df_solar[historical_df_solar["plant_id"] == "PLANT_001"].tail()

,plant_id,plant_type,capacity_mw,timestamp,actual_generation_mw,availability_mw,curtailment_mw,health_factor,status,temperature,...,plant_type_Solar,plant_type_Wind,generation,capacity_factor,generation_shortfall_mw,net_availability_mw,health_adjusted_capacity_mw,is_degraded,is_offline,generation_norm
11492,PLANT_001,Solar,102,2025-04-23 20:00:00,0.0,79.900002,0.00,0.783,ON,33.400002,...,1,0,0.0,0.0,79.900002,79.900002,79.865997,0,0,0.0
11493,PLANT_001,Solar,102,2025-04-23 21:00:00,0.0,79.900002,0.00,0.783,ON,32.400002,...,1,0,0.0,0.0,79.900002,79.900002,79.865997,0,0,0.0
11494,PLANT_001,Solar,102,2025-04-23 22:00:00,0.0,79.900002,0.00,0.783,ON,31.200001,...,1,0,0.0,0.0,79.900002,79.900002,79.865997,0,0,0.0
11495,PLANT_001,Solar,102,2025-04-23 23:00:00,0.0,79.900002,1.19,0.783,ON,30.299999,...,1,0,0.0,0.0,79.900002,78.709999,79.865997,0,0,0.0
11496,PLANT_001,Solar,102,2025-04-24 00:00:00,0.0,79.900002,0.00,0.783,ON,30.299999,...,1,0,0.0,0.0,79.900002,79.900002,79.865997,0,0,0.0


## Derived Features

In [5]:

from src.features import build_features

historical_df_solar = build_features(historical_df_solar, "Solar")
historical_df_wind = build_features(historical_df_wind, "Wind")
historical_df_hybrid = build_features(historical_df_hybrid, "Hybrid")

historical_df_solar.drop(columns=['plant_type'], inplace=True)
historical_df_wind.drop(columns=['plant_type'], inplace=True)
historical_df_hybrid.drop(columns=['plant_type'], inplace=True)


forecast_df_solar = build_features(forecast_df_solar, "Solar")
forecast_df_wind = build_features(forecast_df_wind, "Wind")
forecast_df_hybrid = build_features(forecast_df_hybrid, "Hybrid")

forecast_df_solar.drop(columns=['plant_type'], inplace=True)
forecast_df_wind.drop(columns=['plant_type'], inplace=True)
forecast_df_hybrid.drop(columns=['plant_type'], inplace=True)

In [8]:
historical_df_solar.columns

Index(['plant_id', 'capacity_mw', 'timestamp', 'actual_generation_mw',
       'availability_mw', 'curtailment_mw', 'health_factor', 'temperature',
       'cloud_cover', 'wind_speed', 'wind_direction', 'irradiance', 'hour',
       'day_of_year', 'month', 'day_of_week', 'is_weekend', 'hour_sin',
       'hour_cos', 'doy_sin', 'doy_cos', 'month_sin', 'month_cos',
       'clear_sky_irradiance', 'irradiance_adjusted', 'irradiance_ratio',
       'temp_effect', 'expected_generation', 'days_since_cleaning',
       'soiling_loss', 'adjusted_generation_signal', 'is_daylight'],
      dtype='str')

In [7]:
forecast_df_solar.columns

Index(['plant_id', 'capacity_mw', 'timestamp', 'curtailment_mw',
       'curtail_amount_given_event', 'temperature', 'cloud_cover',
       'wind_speed', 'wind_direction', 'irradiance', 'health_factor',
       'availability_mw', 'plant_type_code', 'plant_type_Hybrid',
       'plant_type_Solar', 'plant_type_Wind', 'hour', 'day_of_year', 'month',
       'day_of_week', 'is_weekend', 'hour_sin', 'hour_cos', 'doy_sin',
       'doy_cos', 'month_sin', 'month_cos', 'clear_sky_irradiance',
       'irradiance_adjusted', 'irradiance_ratio', 'temp_effect',
       'expected_generation', 'days_since_cleaning', 'soiling_loss',
       'adjusted_generation_signal', 'is_daylight'],
      dtype='str')

## Forecasting

In [6]:
from src.multivariate import (
    run_multivariate_fleet,   # full fleet: train + forecast + simulate
    run_single_plant,         # one plant: debug / inspect
    run_decomposition_only,   # STL decomposition fleet-wide (no training)
    decompose_series,         # STL for one plant
    simulate_scenarios,       # Monte Carlo standalone
    select_best_multivariate_model,  # train + forecast one plant (returns dict)
    merge_for_multivariate,   # stack historical + forecast rows
)


In [7]:
all_forecasts_df, all_simulations_df = run_multivariate_fleet(
    historical_df = historical_df_solar,
    forecast_df   = forecast_df_solar,
    feature_cols  = None,      # None → auto-select from EXOGENOUS + ENDOGENOUS
    val_days      = 14,        # walk-forward fold size in days
    n_splits      = 3,         # number of walk-forward folds
    n_simulations = 1000,      # Monte Carlo paths per plant
    n_jobs        = -1,        # -1 = use all CPU cores
    output_dir    = "data/multivariate/solar",
)


Merging DataFrames...
[Merge] historical=172,455  forecast=1,080  plants=15
[Merge] Extended: 172,455 historical + 1,080 forecast rows

════════════════════════════════════════════════════════════
  MULTIVARIATE FLEET RUN
  Plants      : 15
  Target      : generation  (MW)
  Horizon     : t+1 .. t+72h
  Val window  : 14 days × 3 folds
  MC paths    : 1000 / plant
  Workers     : 8 / 8 cores
════════════════════════════════════════════════════════════



[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done   4 out of  15 | elapsed:  3.2min remaining:  8.9min
[Parallel(n_jobs=8)]: Done   8 out of  15 | elapsed:  3.4min remaining:  3.0min
[Parallel(n_jobs=8)]: Done  12 out of  15 | elapsed:  6.0min remaining:  1.5min


  ✓ PLANT_001  model=ridge  MAE=0.019
  ✓ PLANT_004  model=ridge  MAE=0.04
  ✓ PLANT_006  model=ridge  MAE=0.003
  ✓ PLANT_008  model=ridge  MAE=0.009
  ✓ PLANT_014  model=ridge  MAE=0.011
  ✓ PLANT_016  model=ridge  MAE=0.003
  ✓ PLANT_018  model=ridge  MAE=0.01
  ✓ PLANT_026  model=ridge  MAE=0.008
  ✓ PLANT_028  model=ridge  MAE=0.028
  ✓ PLANT_030  model=ridge  MAE=0.007
  ✓ PLANT_035  model=ridge  MAE=0.02
  ✓ PLANT_037  model=ridge  MAE=0.015
  ✓ PLANT_039  model=ridge  MAE=0.012
  ✓ PLANT_041  model=ridge  MAE=0.015
  ✓ PLANT_049  model=ridge  MAE=0.014

════════════════════════════════════════════════════════════
  FLEET COMPLETE  15/15 succeeded

  Model selection:
    ridge       : 15 plants (100%)
════════════════════════════════════════════════════════════

  Saved to data\multivariate\solar/


[Parallel(n_jobs=8)]: Done  15 out of  15 | elapsed:  6.1min finished


In [8]:
all_forecasts_df, all_simulations_df = run_multivariate_fleet(
    historical_df = historical_df_wind,
    forecast_df   = forecast_df_wind,
    feature_cols  = None,      # None → auto-select from EXOGENOUS + ENDOGENOUS
    val_days      = 14,        # walk-forward fold size in days
    n_splits      = 3,         # number of walk-forward folds
    n_simulations = 1000,      # Monte Carlo paths per plant
    n_jobs        = -1,        # -1 = use all CPU cores
    output_dir    = "data/multivariate/wind",
)


Merging DataFrames...
  [WARN] PLANT_015: in historical_df but no forecast rows
[Merge] historical=149,456  forecast=859  plants=13
[Merge] Extended: 149,456 historical + 859 forecast rows

════════════════════════════════════════════════════════════
  MULTIVARIATE FLEET RUN
  Plants      : 13
  Target      : generation  (MW)
  Horizon     : t+1 .. t+72h
  Val window  : 14 days × 3 folds
  MC paths    : 1000 / plant
  Workers     : 8 / 8 cores
════════════════════════════════════════════════════════════



[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done   4 out of  13 | elapsed:  3.1min remaining:  7.0min
[Parallel(n_jobs=8)]: Done   7 out of  13 | elapsed:  3.3min remaining:  2.8min
[Parallel(n_jobs=8)]: Done  10 out of  13 | elapsed:  5.2min remaining:  1.5min


  ✓ PLANT_002  model=ridge  MAE=0.004
  ✓ PLANT_005  model=ridge  MAE=0.005
  ✓ PLANT_007  model=ridge  MAE=0.005
  ✗ PLANT_015  ValueError: Too few training rows: 0 (need ≥500)
  ✓ PLANT_017  model=ridge  MAE=0.002
  ✓ PLANT_027  model=ridge  MAE=0.007
  ✓ PLANT_029  model=ridge  MAE=0.004
  ✓ PLANT_031  model=ridge  MAE=0.008
  ✓ PLANT_036  model=ridge  MAE=0.008
  ✓ PLANT_038  model=ridge  MAE=0.006
  ✓ PLANT_040  model=ridge  MAE=0.004
  ✓ PLANT_048  model=ridge  MAE=0.004
  ✓ PLANT_050  model=ridge  MAE=0.002

════════════════════════════════════════════════════════════
  FLEET COMPLETE  12/13 succeeded

  Model selection:
    ridge       : 12 plants (100%)
════════════════════════════════════════════════════════════

  Saved to data\multivariate\wind/


[Parallel(n_jobs=8)]: Done  13 out of  13 | elapsed:  5.3min finished


In [9]:
all_forecasts_df, all_simulations_df = run_multivariate_fleet(
    historical_df = historical_df_hybrid,
    forecast_df   = forecast_df_hybrid,
    feature_cols  = None,      # None → auto-select from EXOGENOUS + ENDOGENOUS
    val_days      = 14,        # walk-forward fold size in days
    n_splits      = 3,         # number of walk-forward folds
    n_simulations = 1000,      # Monte Carlo paths per plant
    n_jobs        = -1,        # -1 = use all CPU cores
    output_dir    = "data/multivariate/hybrid",
)


Merging DataFrames...
  [WARN] PLANT_013: in historical_df but no forecast rows
[Merge] historical=22,989  forecast=67  plants=2
[Merge] Extended: 22,989 historical + 67 forecast rows

════════════════════════════════════════════════════════════
  MULTIVARIATE FLEET RUN
  Plants      : 2
  Target      : generation  (MW)
  Horizon     : t+1 .. t+72h
  Val window  : 14 days × 3 folds
  MC paths    : 1000 / plant
  Workers     : 2 / 8 cores
════════════════════════════════════════════════════════════



[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.


  ✓ PLANT_003  model=ridge  MAE=0.005
  ✗ PLANT_013  ValueError: Too few training rows: 0 (need ≥500)

════════════════════════════════════════════════════════════
  FLEET COMPLETE  1/2 succeeded

  Model selection:
    ridge       : 1 plants (100%)
════════════════════════════════════════════════════════════

  Saved to data\multivariate\hybrid/


[Parallel(n_jobs=2)]: Done   2 out of   2 | elapsed:  1.4min finished


In [10]:
stl_summary = run_decomposition_only(
    historical_df=historical_df,
    output_csv="data/multivariate/stl_fleet_summary.csv",
)

stl_summary

[PLANT_001] STL:
  daily seasonal : 0.960  (strong)
  weekly seasonal: 0.957  (strong)
  trend          : 0.022
  residual std   : 3.940 MW
  dominant       : daily  |  difficulty: easy
[PLANT_002] STL:
  daily seasonal : 0.216  (weak)
  weekly seasonal: 0.040  (weak)
  trend          : 0.420
  residual std   : 7.658 MW
  dominant       : daily  |  difficulty: hard
[PLANT_003] STL:
  daily seasonal : 0.909  (strong)
  weekly seasonal: 0.888  (strong)
  trend          : 0.319
  residual std   : 2.779 MW
  dominant       : daily  |  difficulty: easy
[PLANT_004] STL:
  daily seasonal : 0.960  (strong)
  weekly seasonal: 0.957  (strong)
  trend          : 0.020
  residual std   : 5.513 MW
  dominant       : daily  |  difficulty: medium
[PLANT_005] STL:
  daily seasonal : 0.217  (weak)
  weekly seasonal: 0.048  (weak)
  trend          : 0.532
  residual std   : 5.130 MW
  dominant       : daily  |  difficulty: hard
[PLANT_006] STL:
  daily seasonal : 0.952  (strong)
  weekly seasonal: 0.948

,plant_id,seasonal_strength,weekly_strength,trend_strength,residual_std,dominant_period,forecast_difficulty
0,PLANT_038,0.2219,0.0480,0.4380,10.5055,daily,hard
1,PLANT_011,0.2187,0.0467,0.4231,8.0472,daily,hard
2,PLANT_046,0.2097,0.0405,0.4186,8.0464,daily,hard
3,PLANT_029,0.2167,0.0542,0.4257,7.9672,daily,hard
4,PLANT_025,0.2135,0.0476,0.4317,7.7650,daily,hard
5,PLANT_007,0.2115,0.0450,0.4396,7.7291,daily,hard
6,PLANT_002,0.2161,0.0398,0.4200,7.6582,daily,hard
7,PLANT_031,0.2174,0.0403,0.3862,7.6116,daily,hard
8,PLANT_028,0.9586,0.9564,0.0205,7.1935,daily,medium
9,PLANT_021,0.2245,0.0471,0.4351,6.9072,daily,hard


In [ ]:

# Inspect difficulty distribution
print("\nFleet difficulty breakdown:")
print(stl_summary["forecast_difficulty"].value_counts())

# Focus on hard plants that need more attention
hard_plants = stl_summary[stl_summary["forecast_difficulty"] == "hard"]
print(f"\nHard plants ({len(hard_plants)}):")
print(hard_plants[["plant_id", "residual_std", "seasonal_strength"]].to_string(index=False))